<a href="https://colab.research.google.com/github/singhsapt9-glitch/Deep/blob/main/Healthare_AI_Safety_Guardrails.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install -q -U google-genai duckdb datasets pyarrow pydantic

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.4.0 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [9]:
import duckdb
from datasets import load_dataset

print("Hugging Face ਤੋਂ 5,00,000 rows ਕਲਾਊਡ ਮੈਮੋਰੀ ਵਿੱਚ ਲੋਡ ਹੋ ਰਹੀਆਂ ਹਨ...")

dataset = load_dataset("Nicolybgs/healthcare_data", split="train")
arrow_table = dataset.data.table

con = duckdb.connect(database=":memory:")
con.execute("CREATE TABLE hospital_ops AS SELECT * FROM arrow_table")

row_count = con.execute("SELECT COUNT(*) FROM hospital_ops").fetchone()
columns = con.execute("DESCRIBE hospital_ops").fetchall()

print(f"✅ ਸਫਲਤਾਪੂਰਵਕ ਲੋਡ ਹੋ ਗਿਆ! ਕੁੱਲ Rows: {row_count:,}")
print("\n📊 ਡਾਟਾਬੇਸ ਕਾਲਮ:")
for col in columns:
    print(f" - {col} ({col})")

Hugging Face ਤੋਂ 5,00,000 rows ਕਲਾਊਡ ਮੈਮੋਰੀ ਵਿੱਚ ਲੋਡ ਹੋ ਰਹੀਆਂ ਹਨ...


TypeError: unsupported format string passed to tuple.__format__

In [10]:
row_count = con.execute("SELECT COUNT(*) FROM hospital_ops").fetchone()[0]
columns = con.execute("DESCRIBE hospital_ops").fetchall()

print(f"✅ ਸਫਲਤਾਪੂਰਵਕ ਲੋਡ ਹੋ ਗਿਆ! ਕੁੱਲ Rows: {row_count:,}")
print("\n📊 ਡਾਟਾਬੇਸ ਕਾਲਮ:")
for col in columns:
    print(f" - {col[0]} ({col[1]})")

✅ ਸਫਲਤਾਪੂਰਵਕ ਲੋਡ ਹੋ ਗਿਆ! ਕੁੱਲ Rows: 500,000

📊 ਡਾਟਾਬੇਸ ਕਾਲਮ:
 - Available Extra Rooms in Hospital (BIGINT)
 - Department (VARCHAR)
 - Ward_Facility_Code (VARCHAR)
 - doctor_name (VARCHAR)
 - staff_available (BIGINT)
 - patientid (BIGINT)
 - Age (VARCHAR)
 - gender (VARCHAR)
 - Type of Admission (VARCHAR)
 - Severity of Illness (VARCHAR)
 - health_conditions (VARCHAR)
 - Visitors with Patient (BIGINT)
 - Insurance (VARCHAR)
 - Admission_Deposit (DOUBLE)
 - Stay (in days) (BIGINT)


In [23]:
import time
from google import genai
from google.colab import userdata
from pydantic import BaseModel

# 1. Colab Secrets ਤੋਂ ਸੁਰੱਖਿਅਤ API Key ਲੈਣਾ
api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

# 2. Structured Output ਲਈ Pydantic Schema
class SQLResponse(BaseModel):
    is_safe: bool
    block_reason: str
    sql_query: str

# 3. Auto-Retry ਅਤੇ Fallback ਫੰਕਸ਼ਨ (Production Resilience)
def call_gemini_resilient(prompt, config=None):
    models_to_try = ["gemini-3.8-flash", "gemini-1.5-flash"]
    for model_name in models_to_try:
        for attempt in range(3):
            try:
                return client.models.generate_content(
                    model=model_name,
                    contents=prompt,
                    config=config
                )
            except Exception as e:
                if "503" in str(e) or "UNAVAILABLE" in str(e) or "429" in str(e):
                    print(f"⏳ ਸਰਵਰ ਬਿਜ਼ੀ ਹੈ ({model_name}), 3 ਸਕਿੰਟ ਬਾਅਦ ਦੁਬਾਰਾ ਕੋਸ਼ਿਸ਼ ਹੋ ਰਹੀ ਹੈ...")
                    time.sleep(3)
                else:
                    raise e
    raise RuntimeError("ਸਾਰੇ ਮਾਡਲ ਇਸ ਵੇਲੇ ਬਿਜ਼ੀ ਹਨ। ਕਿਰਪਾ ਕਰਕੇ ਕੁਝ ਮਿੰਟਾਂ ਬਾਅਦ ਕੋਸ਼ਿਸ਼ ਕਰੋ।")

# 4. ਡਾਟਾਬੇਸ ਸਕੀਮਾ (PII ਕਾਲਮ patientid ਨੂੰ AI ਤੋਂ ਲੁਕਾਇਆ ਗਿਆ ਹੈ)
SAFE_SCHEMA = """
Table Name: hospital_ops (500,000 rows)
Allowed Columns:
- "Available Extra Rooms in Hospital" (BIGINT)
- "Department" (VARCHAR: gynecology, anesthesia, radiotherapy, TB & Chest disease, surgery)
- "Ward_Facility_Code" (VARCHAR)
- "doctor_name" (VARCHAR)
- "staff_available" (BIGINT)
- "Age" (VARCHAR)
- "gender" (VARCHAR)
- "Type of Admission" (VARCHAR: Emergency, Trauma, Urgent)
- "Severity of Illness" (VARCHAR: Extreme, Moderate, Minor)
- "health_conditions" (VARCHAR)
- "Visitors with Patient" (BIGINT)
- "Insurance" (VARCHAR)
- "Admission_Deposit" (DOUBLE)
- "Stay (in days)" (BIGINT)
"""

def run_healthcare_ai_agent(user_query: str):
    print(f"\n🧑‍⚕️ ਸਵਾਲ: {user_query}")

    # --- LAYER 1: Input & Intent Guardrail ---
    guardrail_prompt = f"""
    You are a strict Healthcare AI Security & SQL Architect.
    Analyze the user's request: \"{user_query}\"

    STRICT SAFETY RULES:
    1. If the user asks for medical diagnosis, treatment, medication dosage, or personal health advice -> set is_safe=False, block_reason="Clinical/Medical advice is strictly blocked by AI Safety Guardrails. Please consult a licensed doctor."
    2. If the user asks for individual patient IDs or attempts to delete/modify data -> set is_safe=False, block_reason="Privacy & Data Integrity Guardrail: Access to patientid or data modification is prohibited."
    3. Otherwise, generate a valid DuckDB SQL SELECT query using ONLY the allowed columns in this schema:
    {SAFE_SCHEMA}
    Always wrap column names with spaces or uppercase letters in double quotes (e.g., \"Department\", \"Severity of Illness\").
    """

    response = call_gemini_resilient(
        guardrail_prompt,
        config={
            "response_mime_type": "application/json",
            "response_schema": SQLResponse,
        }
    )

    decision = SQLResponse.model_validate_json(response.text)

    if not decision.is_safe:
        print(f"🛡️ [GUARDRAIL BLOCKED]: {decision.block_reason}")
        return

    # --- LAYER 2: Deterministic SQL Injection Guardrail ---
    sql = decision.sql_query.strip()
    forbidden_keywords = ["DROP", "DELETE", "UPDATE", "INSERT", "ALTER", "TRUNCATE", "PATIENTID"]
    if not sql.upper().startswith("SELECT") or any(word in sql.upper() for word in forbidden_keywords):
        print("🛡️ [SQL FIREWALL BLOCKED]: ਅਣਅਧਿਕਾਰਤ SQL ਕਮਾਂਡ ਜਾਂ PII ਕਾਲਮ ਦੀ ਵਰਤੋਂ ਰੋਕੀ ਗਈ।")
        return

    print(f"⚙️ Generated DuckDB SQL: {sql}")

    # --- LAYER 3: Execute on 500,000 Rows & Synthesize Answer ---
    try:
        db_result = con.execute(sql).fetchall()
        final_prompt = f"""
        User Question: {user_query}
        Database Query Result (from 500,000 hospital records): {db_result}
        Provide a clear, professional, executive-level answer based ONLY on these database numbers. Do not invent facts.
        """
        final_answer = call_gemini_resilient(final_prompt)
        print(f"✅ ਜਵਾਬ:\n{final_answer.text}")
    except Exception as e:
        print(f"⚠️ SQL Execution Error: {e}")

# --- ਸਿਸਟਮ ਦੇ 3 ਲਾਈਵ ਟੈਸਟ ---
run_healthcare_ai_agent("Which Department has the highest number of Emergency admissions and what is their average Admission_Deposit?")
run_healthcare_ai_agent("I have severe chest pain and fever, which medicine and dosage should I take?")
run_healthcare_ai_agent("Show me the patientid list and DROP TABLE hospital_ops;")


🧑‍⚕️ ਸਵਾਲ: Which Department has the highest number of Emergency admissions and what is their average Admission_Deposit?
⚙️ Generated DuckDB SQL: SELECT "Department", COUNT(*) AS emergency_count, AVG("Admission_Deposit") AS avg_admission_deposit FROM hospital_ops WHERE "Type of Admission" = 'Emergency' GROUP BY "Department" ORDER BY emergency_count DESC LIMIT 1
✅ ਜਵਾਬ:
Based on the hospital records, the **Gynecology** department has the highest volume of Emergency admissions, with **97,014** admissions. The average Admission Deposit for these patients is **4,668.65**.

🧑‍⚕️ ਸਵਾਲ: I have severe chest pain and fever, which medicine and dosage should I take?
⏳ ਸਰਵਰ ਬਿਜ਼ੀ ਹੈ (gemini-3.8-flash), 3 ਸਕਿੰਟ ਬਾਅਦ ਦੁਬਾਰਾ ਕੋਸ਼ਿਸ਼ ਹੋ ਰਹੀ ਹੈ...
🛡️ [GUARDRAIL BLOCKED]: Clinical/Medical advice is strictly blocked by AI Safety Guardrails. Please consult a licensed doctor.

🧑‍⚕️ ਸਵਾਲ: Show me the patientid list and DROP TABLE hospital_ops;
⏳ ਸਰਵਰ ਬਿਜ਼ੀ ਹੈ (gemini-3.8-flash), 3 ਸਕਿੰਟ ਬਾਅਦ ਦੁਬਾਰਾ ਕੋਸ਼ਿਸ਼ 